# Credit Risk Feature Engineering

## Objective

This notebook transforms the raw Home Credit application data into a modeling-ready dataset.

The objectives are to:

1. Split the data before learning preprocessing parameters
2. Correct invalid and special values
3. Create interpretable credit-risk features
4. Avoid duplicate information and unnecessary multicollinearity
5. Define numerical and categorical feature groups
6. Build a reproducible preprocessing workflow
7. Save the fitted preprocessing artifacts for baseline modeling

This notebook focuses on feature engineering and preprocessing. Model training and evaluation will be completed in the next notebook.

## Modeling Principles

- Split the data before fitting imputers, encoders, or scalers
- Use only information available at application time
- Preserve missingness when it may carry risk information
- Handle special values inside reusable transformation functions
- Handle division-by-zero explicitly
- Avoid retaining exact linear duplicates of engineered features
- Apply identical transformations to training and test data
- Fit all learned preprocessing parameters using training data only

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

print("Python version:", sys.version)

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

TRAIN_PATH = RAW_DATA_DIR / "application_train.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_PATH)

print("Dataset shape:", df.shape)
df.head()

## 2. Basic Validation

In [ ]:
required_columns = ["SK_ID_CURR", "TARGET"]

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {missing_required_columns}"
    )

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique applicants:", df["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", df["SK_ID_CURR"].duplicated().sum())
print("\nTarget distribution:")
print(df["TARGET"].value_counts(dropna=False))
print("\nTarget rate:")
print(df["TARGET"].mean())

In [ ]:
assert df["SK_ID_CURR"].duplicated().sum() == 0
assert set(df["TARGET"].dropna().unique()).issubset({0, 1})

### Observation

- The application table contains one row per applicant.
- `SK_ID_CURR` is an identifier and should not be used as a model feature.
- `TARGET = 1` represents applicants with repayment difficulties.
- The target is imbalanced, so accuracy alone will not be sufficient for model evaluation.

## 3. Train/Test Split

In [ ]:
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Training target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

In [ ]:
assert len(X_train) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert abs(y_train.mean() - y_test.mean()) < 0.001

In [ ]:
train_ids = X_train["SK_ID_CURR"].copy()
test_ids = X_test["SK_ID_CURR"].copy()

X_train = X_train.drop(columns=["SK_ID_CURR"])
X_test = X_test.drop(columns=["SK_ID_CURR"])

## 4. Initial Feature Scope

In [ ]:
initial_features = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

missing_initial_features = [
    col for col in initial_features
    if col not in X_train.columns
]

if missing_initial_features:
    raise ValueError(
        f"Initial features missing from dataset: {missing_initial_features}"
    )

X_train_fe = X_train[initial_features].copy()
X_test_fe = X_test[initial_features].copy()

print("Initial feature count:", len(initial_features))
print("Training feature dataset:", X_train_fe.shape)
print("Test feature dataset:", X_test_fe.shape)

In [ ]:
feature_summary = pd.DataFrame({
    "dtype": X_train_fe.dtypes.astype(str),
    "missing_count": X_train_fe.isna().sum(),
    "missing_rate": X_train_fe.isna().mean(),
    "n_unique": X_train_fe.nunique(dropna=False),
}).sort_values("missing_rate", ascending=False)

feature_summary.head(20)

## 5. Reusable Domain Feature Engineering

In [ ]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series
) -> pd.Series:
    """Safely divide two pandas Series."""
    denominator_clean = denominator.replace(0, np.nan)
    result = numerator / denominator_clean
    return result.replace([np.inf, -np.inf], np.nan)

In [ ]:
def create_domain_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Create interpretable credit-risk features from raw application data.

    This function includes all deterministic cleaning required before the
    learned sklearn preprocessing pipeline is applied.
    """
    data = data.copy()

    # --------------------------------------------------
    # Special-value cleaning
    # --------------------------------------------------
    special_employed_value = 365243

    data["DAYS_EMPLOYED_SPECIAL_FLAG"] = (
        data["DAYS_EMPLOYED"] == special_employed_value
    ).astype("int8")

    data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(
        special_employed_value,
        np.nan
    )

    # --------------------------------------------------
    # Applicant age and employment history
    # --------------------------------------------------
    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
    data["EMPLOYMENT_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25
    data["REGISTRATION_YEARS"] = -data["DAYS_REGISTRATION"] / 365.25
    data["ID_PUBLISH_YEARS"] = -data["DAYS_ID_PUBLISH"] / 365.25

    data.loc[
        (data["AGE_YEARS"] < 18) | (data["AGE_YEARS"] > 100),
        "AGE_YEARS"
    ] = np.nan

    data.loc[
        data["EMPLOYMENT_YEARS"] < 0,
        "EMPLOYMENT_YEARS"
    ] = np.nan

    data.loc[
        data["EMPLOYMENT_YEARS"] > data["AGE_YEARS"],
        "EMPLOYMENT_YEARS"
    ] = np.nan

    # --------------------------------------------------
    # Affordability features
    # --------------------------------------------------
    data["CREDIT_INCOME_RATIO"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_INCOME_TOTAL"]
    )

    data["ANNUITY_INCOME_RATIO"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_INCOME_TOTAL"]
    )

    data["CREDIT_ANNUITY_RATIO"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_ANNUITY"]
    )

    data["INCOME_PER_PERSON"] = safe_divide(
        data["AMT_INCOME_TOTAL"],
        data["CNT_FAM_MEMBERS"]
    )

    data["CREDIT_PER_PERSON"] = safe_divide(
        data["AMT_CREDIT"],
        data["CNT_FAM_MEMBERS"]
    )

    # --------------------------------------------------
    # Loan structure features
    # --------------------------------------------------
    data["GOODS_CREDIT_RATIO"] = safe_divide(
        data["AMT_GOODS_PRICE"],
        data["AMT_CREDIT"]
    )

    data["CREDIT_GOODS_DIFFERENCE"] = (
        data["AMT_CREDIT"] - data["AMT_GOODS_PRICE"]
    )

    data["ANNUITY_GOODS_RATIO"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_GOODS_PRICE"]
    )

    # --------------------------------------------------
    # Employment stability
    # --------------------------------------------------
    data["EMPLOYMENT_AGE_RATIO"] = safe_divide(
        data["EMPLOYMENT_YEARS"],
        data["AGE_YEARS"]
    )

    # --------------------------------------------------
    # External scores
    # --------------------------------------------------
    ext_source_cols = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    ]

    data["EXT_SOURCE_MEAN"] = data[ext_source_cols].mean(axis=1)
    data["EXT_SOURCE_MIN"] = data[ext_source_cols].min(axis=1)
    data["EXT_SOURCE_MAX"] = data[ext_source_cols].max(axis=1)
    data["EXT_SOURCE_STD"] = data[ext_source_cols].std(axis=1)

    data["EXT_SOURCE_MISSING_COUNT"] = (
        data[ext_source_cols]
        .isna()
        .sum(axis=1)
        .astype("int8")
    )

    # --------------------------------------------------
    # Bureau inquiries
    # --------------------------------------------------
    bureau_inquiry_cols = [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
        "AMT_REQ_CREDIT_BUREAU_QRT",
        "AMT_REQ_CREDIT_BUREAU_YEAR",
    ]

    data["BUREAU_INQUIRY_TOTAL"] = (
        data[bureau_inquiry_cols]
        .sum(axis=1, min_count=1)
    )

    recent_inquiry_cols = [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
    ]

    data["BUREAU_INQUIRY_RECENT"] = (
        data[recent_inquiry_cols]
        .sum(axis=1, min_count=1)
    )

    # --------------------------------------------------
    # Remove exact linear duplicates of year features
    # --------------------------------------------------
    raw_time_features_to_drop = [
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
    ]

    data = data.drop(columns=raw_time_features_to_drop)

    # --------------------------------------------------
    # Applicant-level missingness
    # --------------------------------------------------
    data["TOTAL_MISSING_COUNT"] = (
        data.isna()
        .sum(axis=1)
        .astype("int16")
    )

    data["TOTAL_MISSING_RATE"] = data.isna().mean(axis=1)

    return data

In [ ]:
X_train_fe = create_domain_features(X_train_fe)
X_test_fe = create_domain_features(X_test_fe)

print("Training shape after feature engineering:", X_train_fe.shape)
print("Test shape after feature engineering:", X_test_fe.shape)

## 6. Feature Engineering Validation

In [ ]:
assert list(X_train_fe.columns) == list(X_test_fe.columns)

numeric_train = X_train_fe.select_dtypes(include=["number"])
numeric_test = X_test_fe.select_dtypes(include=["number"])

train_infinite_count = np.isinf(numeric_train).sum().sum()
test_infinite_count = np.isinf(numeric_test).sum().sum()

print("Infinite values in training data:", train_infinite_count)
print("Infinite values in test data:", test_infinite_count)

assert train_infinite_count == 0
assert test_infinite_count == 0

In [ ]:
validation_summary = pd.DataFrame({
    "dtype": X_train_fe.dtypes.astype(str),
    "missing_count": X_train_fe.isna().sum(),
    "missing_rate": X_train_fe.isna().mean(),
    "n_unique": X_train_fe.nunique(dropna=False),
}).sort_values("missing_rate", ascending=False)

validation_summary.head(25)

In [ ]:
engineered_features = [
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "REGISTRATION_YEARS",
    "ID_PUBLISH_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_ANNUITY_RATIO",
    "INCOME_PER_PERSON",
    "CREDIT_PER_PERSON",
    "GOODS_CREDIT_RATIO",
    "CREDIT_GOODS_DIFFERENCE",
    "ANNUITY_GOODS_RATIO",
    "EMPLOYMENT_AGE_RATIO",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_MISSING_COUNT",
    "BUREAU_INQUIRY_TOTAL",
    "BUREAU_INQUIRY_RECENT",
    "TOTAL_MISSING_COUNT",
    "TOTAL_MISSING_RATE",
]

X_train_fe[engineered_features].describe().T[
    ["count", "mean", "std", "min", "50%", "max"]
].round(3)

## 7. Descriptive Target Comparison

In [ ]:
train_feature_review = X_train_fe.copy()
train_feature_review["TARGET"] = y_train

feature_target_review = (
    train_feature_review
    .groupby("TARGET")[engineered_features]
    .mean()
    .T
)

feature_target_review.columns = [
    "TARGET_0_MEAN",
    "TARGET_1_MEAN",
]

feature_target_review["ABSOLUTE_DIFFERENCE"] = (
    feature_target_review["TARGET_1_MEAN"]
    - feature_target_review["TARGET_0_MEAN"]
)

feature_target_review.sort_values(
    by="ABSOLUTE_DIFFERENCE",
    key=lambda series: series.abs(),
    ascending=False
)

## 8. Feature-Type Definition

In [ ]:
categorical_features = (
    X_train_fe
    .select_dtypes(include=["object", "string", "category"])
    .columns
    .tolist()
)

numerical_features = (
    X_train_fe
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

classified_features = set(
    categorical_features + numerical_features
)

unclassified_features = (
    set(X_train_fe.columns) - classified_features
)

print("Number of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))
print("Unclassified features:", unclassified_features)

assert not unclassified_features

In [ ]:
categorical_cardinality = (
    X_train_fe[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .to_frame("n_unique")
)

categorical_cardinality

### High-Cardinality Note

`ORGANIZATION_TYPE` may exceed the initial high-cardinality threshold, but it is retained for the baseline because the final encoded feature space remains manageable. Rare-category grouping can be evaluated later if it improves stability or interpretability.

## 9. Preprocessing Pipeline

In [ ]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

In [ ]:
processed_feature_names = preprocessor.get_feature_names_out()

clean_feature_names = [
    feature_name
    .replace("numerical__", "")
    .replace("categorical__", "")
    for feature_name in processed_feature_names
]

print("Number of processed features:", len(clean_feature_names))
clean_feature_names[:30]

## 10. Transformation Validation

In [ ]:
def count_missing_and_infinite(matrix):
    values = matrix.data if hasattr(matrix, "data") else matrix
    return {
        "missing": int(np.isnan(values).sum()),
        "infinite": int(np.isinf(values).sum()),
    }

train_validation = count_missing_and_infinite(X_train_processed)
test_validation = count_missing_and_infinite(X_test_processed)

print("Training:", train_validation)
print("Test:", test_validation)

assert train_validation["missing"] == 0
assert test_validation["missing"] == 0
assert train_validation["infinite"] == 0
assert test_validation["infinite"] == 0

## 11. Save Preprocessing Artifacts

The saved sklearn preprocessor expects data after `create_domain_features()` has been applied.

Correct usage for new raw application data:

```python
new_data_fe = create_domain_features(new_data)
new_data_processed = preprocessor.transform(new_data_fe)
```

The deterministic feature-engineering function and the learned sklearn preprocessor must both be preserved for future scoring.

In [ ]:
PREPROCESSOR_PATH = MODEL_DIR / "feature_preprocessor.joblib"
FEATURE_NAMES_PATH = MODEL_DIR / "processed_feature_names.txt"

joblib.dump(preprocessor, PREPROCESSOR_PATH)

with open(FEATURE_NAMES_PATH, "w", encoding="utf-8") as file:
    for feature_name in clean_feature_names:
        file.write(f"{feature_name}\n")

print("Saved preprocessor to:", PREPROCESSOR_PATH)
print("Saved feature names to:", FEATURE_NAMES_PATH)

In [ ]:
train_split_output = pd.DataFrame({
    "SK_ID_CURR": train_ids,
    "TARGET": y_train,
}).reset_index(drop=True)

test_split_output = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "TARGET": y_test,
}).reset_index(drop=True)

TRAIN_SPLIT_PATH = (
    PROCESSED_DATA_DIR / "train_split_ids_targets.csv"
)

TEST_SPLIT_PATH = (
    PROCESSED_DATA_DIR / "test_split_ids_targets.csv"
)

train_split_output.to_csv(TRAIN_SPLIT_PATH, index=False)
test_split_output.to_csv(TEST_SPLIT_PATH, index=False)

print("Saved:", TRAIN_SPLIT_PATH)
print("Saved:", TEST_SPLIT_PATH)

## 12. Final Summary

In [ ]:
preprocessing_summary = pd.Series({
    "training_rows": X_train_fe.shape[0],
    "test_rows": X_test_fe.shape[0],
    "raw_feature_count_after_engineering": X_train_fe.shape[1],
    "categorical_feature_count": len(categorical_features),
    "numerical_feature_count": len(numerical_features),
    "processed_feature_count": X_train_processed.shape[1],
    "training_target_rate": y_train.mean(),
    "test_target_rate": y_test.mean(),
    "training_missing_after_processing": train_validation["missing"],
    "test_missing_after_processing": test_validation["missing"],
    "training_infinite_after_processing": train_validation["infinite"],
    "test_infinite_after_processing": test_validation["infinite"],
})

preprocessing_summary

## Preprocessing Summary

The workflow now:

- performs a stratified train/test split before learning preprocessing parameters,
- removes the applicant identifier from model features,
- handles `DAYS_EMPLOYED = 365243` inside the reusable feature function,
- creates interpretable credit-risk and affordability features,
- removes exact linear duplicates of converted time variables,
- preserves informative missingness,
- uses median imputation and missing indicators for numerical variables,
- uses most-frequent imputation and one-hot encoding for categorical variables,
- safely handles unseen categories,
- confirms that no missing or infinite values remain,
- saves the fitted preprocessor and feature names.

The next notebook can build and evaluate an interpretable logistic regression baseline.